In [1]:
import pandas as pd
import numpy as np
from pathlib import Path
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from tqdm import tqdm

PHASE0_DIR = Path("/kaggle/input/datasets/<username>/rsna-atd-phase0-preprocessed/processed")

MANIFEST_PATH = PHASE0_DIR / "manifest.csv"
LABELS_PATH = PHASE0_DIR / "subset_labels.csv"

OUTPUT_DIR = Path("/kaggle/working")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

TARGET_SIZE = 256
N_SLICES_PER_ORGAN = 32

In [2]:
print("PHASE0_DIR exists:", PHASE0_DIR.exists())
print("Manifest exists:", MANIFEST_PATH.exists())
print("Labels exists:", LABELS_PATH.exists())

PHASE0_DIR exists: True
Manifest exists: True
Labels exists: True


In [3]:
manifest_df = pd.read_csv(MANIFEST_PATH)

manifest_df['output_path'] = manifest_df['output_path'].apply(
    lambda p: str(PHASE0_DIR / Path(p).name)
)

sample = manifest_df.iloc[0]
print("Rewritten path:", sample['output_path'])
print("Exists:", Path(sample['output_path']).exists())

manifest_df['organs_cropped'] = manifest_df['organs_cropped'].apply(eval)
manifest_df['resize_factors'] = manifest_df['resize_factors'].apply(eval)

print(f"\nTotal series: {len(manifest_df)}")
print(f"Series with organ crops: {(manifest_df['organs_cropped'].apply(len) > 0).sum()}")

Rewritten path: /kaggle/input/datasets/<username>/rsna-atd-phase0-preprocessed/processed/10004_21057.npz
Exists: True

Total series: 1803
Series with organ crops: 206


In [4]:
from sklearn.model_selection import train_test_split

labels_df = pd.read_csv(LABELS_PATH)

strat_df = labels_df[['patient_id', 'any_injury', 'bowel_injury', 'extravasation_injury',
                       'kidney_high', 'liver_high', 'spleen_high']].copy()
strat_df['strat_key'] = (
    strat_df['bowel_injury'].astype(str) + "_" +
    strat_df['extravasation_injury'].astype(str) + "_" +
    (strat_df[['kidney_high','liver_high','spleen_high']].sum(axis=1) > 0).astype(int).astype(str)
)

bucket_counts = strat_df['strat_key'].value_counts()
rare_buckets = bucket_counts[bucket_counts < 2].index
strat_df.loc[strat_df['strat_key'].isin(rare_buckets), 'strat_key'] = "other"

train_patients, val_patients = train_test_split(
    strat_df['patient_id'],
    test_size=0.15,
    random_state=42,
    stratify=strat_df['strat_key']
)
train_patients, val_patients = set(train_patients), set(val_patients)

print(f"Train patients: {len(train_patients)}, Val patients: {len(val_patients)}")

manifest_labeled = manifest_df.merge(labels_df, on='patient_id', how='left')
manifest_labeled['split'] = manifest_labeled['patient_id'].apply(
    lambda p: 'train' if p in train_patients else ('val' if p in val_patients else 'unused')
)
organ_manifest = manifest_labeled[manifest_labeled['organs_cropped'].apply(len) > 0].reset_index(drop=True)

print(f"\nSeries usable for organ-branch training: {len(organ_manifest)}")
print(organ_manifest['split'].value_counts())
print(organ_manifest.groupby('split')['any_injury'].mean())

organ_manifest.to_csv(OUTPUT_DIR / "organ_manifest_split.csv", index=False)

Train patients: 1020, Val patients: 180

Series usable for organ-branch training: 206
split
train    173
val       33
Name: count, dtype: int64
split
train    1.0
val      1.0
Name: any_injury, dtype: float64


In [5]:
print(organ_manifest.shape)
print(organ_manifest['split'].value_counts())
print(organ_manifest.columns.tolist())

(206, 23)
split
train    173
val       33
Name: count, dtype: int64
['patient_id', 'series_id', 'has_segmentation', 'organs_cropped', 'resize_factors', 'has_full_volume', 'output_path', 'bowel_healthy', 'bowel_injury', 'extravasation_healthy', 'extravasation_injury', 'kidney_healthy', 'kidney_low', 'kidney_high', 'liver_healthy', 'liver_low', 'liver_high', 'spleen_healthy', 'spleen_low', 'spleen_high', 'any_injury', 'has_seg', 'split']


In [6]:
def _peek_label(row, organ):
    if organ == "bowel":
        return int(row['bowel_injury'])
    h, l, hi = row[f'{organ}_healthy'], row[f'{organ}_low'], row[f'{organ}_high']
    return int(np.argmax([h, l, hi]))

for organ in ["liver", "spleen", "kidney", "bowel"]:
    sub = organ_manifest[organ_manifest['organs_cropped'].apply(lambda x: organ in x)].copy()
    sub['label'] = sub.apply(lambda row: _peek_label(row, organ), axis=1)

    print(f"\n=== {organ} (n={len(sub)}) ===")
    print("Overall label counts:")
    print(sub['label'].value_counts().sort_index())

    train_sub = sub[sub['split'] == 'train']
    val_sub = sub[sub['split'] == 'val']

    print(f"\nTrain split (n={len(train_sub)}) label counts:")
    print(train_sub['label'].value_counts().sort_index())

    print(f"Val split (n={len(val_sub)}) label counts:")
    print(val_sub['label'].value_counts().sort_index())


=== liver (n=206) ===
Overall label counts:
label
0    83
1    92
2    31
Name: count, dtype: int64

Train split (n=173) label counts:
label
0    69
1    77
2    27
Name: count, dtype: int64
Val split (n=33) label counts:
label
0    14
1    15
2     4
Name: count, dtype: int64

=== spleen (n=206) ===
Overall label counts:
label
0    67
1    59
2    80
Name: count, dtype: int64

Train split (n=173) label counts:
label
0    54
1    52
2    67
Name: count, dtype: int64
Val split (n=33) label counts:
label
0    13
1     7
2    13
Name: count, dtype: int64

=== kidney (n=204) ===
Overall label counts:
label
0    101
1     56
2     47
Name: count, dtype: int64

Train split (n=171) label counts:
label
0    86
1    49
2    36
Name: count, dtype: int64
Val split (n=33) label counts:
label
0    15
1     7
2    11
Name: count, dtype: int64

=== bowel (n=206) ===
Overall label counts:
label
0    184
1     22
Name: count, dtype: int64

Train split (n=173) label counts:
label
0    152
1     21
Name

In [7]:
ORGAN_NUM_CLASSES = {"liver": 3, "spleen": 3, "kidney": 3, "bowel": 2}

def get_organ_label(row, organ):
    if organ == "bowel":
        return int(row['bowel_injury'])
    healthy, low, high = row[f'{organ}_healthy'], row[f'{organ}_low'], row[f'{organ}_high']
    return int(np.argmax([healthy, low, high]))

class OrganCropDataset(Dataset):
    def __init__(self, manifest_df, organ, n_slices=N_SLICES_PER_ORGAN, img_size=TARGET_SIZE):
        self.df = manifest_df[manifest_df['organs_cropped'].apply(lambda x: organ in x)].reset_index(drop=True)
        self.organ = organ
        self.n_slices = n_slices
        self.img_size = img_size

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        data = np.load(row['output_path'])
        vol = data[self.organ].astype(np.float32) / 255.0

        label = get_organ_label(row, self.organ)
        weight = self._get_sample_weight(row)

        vol = torch.from_numpy(vol).unsqueeze(1)
        return vol, label, weight

    def _get_sample_weight(self, row):
        if self.organ == "bowel":
            return 2.0 if row['bowel_injury'] == 1 else 1.0
        healthy, low, high = row[f'{self.organ}_healthy'], row[f'{self.organ}_low'], row[f'{self.organ}_high']
        if high == 1: return 4.0
        if low == 1: return 2.0
        return 1.0

def make_loader(manifest_df, organ, split, batch_size=4, num_workers=2):
    df_split = manifest_df[manifest_df['split'] == split]
    ds = OrganCropDataset(df_split, organ)
    return DataLoader(ds, batch_size=batch_size, shuffle=(split == 'train'),
                       num_workers=num_workers, pin_memory=True)

In [8]:
!pip install -q timm
import timm

class SliceAttentionPool(nn.Module):
    def __init__(self, feat_dim):
        super().__init__()
        self.attn = nn.Sequential(nn.Linear(feat_dim, 128), nn.Tanh(), nn.Linear(128, 1))

    def forward(self, feats):
        weights = torch.softmax(self.attn(feats), dim=1)
        pooled = (feats * weights).sum(dim=1)
        return pooled

class OrganClassifier(nn.Module):
    def __init__(self, num_classes, backbone_name="efficientnet_b0"):
        super().__init__()
        self.backbone = timm.create_model(backbone_name, pretrained=True, num_classes=0, in_chans=1)
        feat_dim = self.backbone.num_features
        self.pool = SliceAttentionPool(feat_dim)
        self.head = nn.Linear(feat_dim, num_classes)

    def forward(self, x):
        B, S, C, H, W = x.shape
        x = x.view(B * S, C, H, W)
        feats = self.backbone(x)
        feats = feats.view(B, S, -1)
        pooled = self.pool(feats)
        return self.head(pooled)

In [9]:
import random

def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

def train_organ_model(organ, epochs=5, lr=1e-4, seed=42):
    set_seed(seed)

    train_loader = make_loader(organ_manifest, organ, 'train')
    val_loader = make_loader(organ_manifest, organ, 'val')

    print(f"[{organ}] train samples: {len(train_loader.dataset)}, val samples: {len(val_loader.dataset)}")

    model = OrganClassifier(num_classes=ORGAN_NUM_CLASSES[organ]).to(device)
    optimizer = optim.AdamW(model.parameters(), lr=lr)
    criterion = nn.CrossEntropyLoss(reduction='none')

    best_val_loss = float('inf')
    history = []
    best_val_predictions = None

    for epoch in range(epochs):
        model.train()
        train_losses = []
        for vol, label, weight in tqdm(train_loader, desc=f"{organ} epoch {epoch+1}/{epochs}"):
            vol, label, weight = vol.to(device), label.to(device), weight.to(device).float()
            optimizer.zero_grad()
            logits = model(vol)
            loss_per_sample = criterion(logits, label)
            loss = (loss_per_sample * weight).mean()
            loss.backward()
            optimizer.step()
            train_losses.append(loss.item())

        model.eval()
        val_losses = []
        val_preds, val_labels = [], []
        with torch.no_grad():
            for vol, label, weight in val_loader:
                vol, label, weight = vol.to(device), label.to(device), weight.to(device).float()
                logits = model(vol)
                loss_per_sample = criterion(logits, label)
                loss = (loss_per_sample * weight).mean()
                val_losses.append(loss.item())
                val_preds.extend(torch.argmax(logits, dim=1).cpu().numpy().tolist())
                val_labels.extend(label.cpu().numpy().tolist())

        train_loss = np.mean(train_losses) if train_losses else float('nan')
        val_loss = np.mean(val_losses) if val_losses else float('nan')
        val_acc = np.mean(np.array(val_preds) == np.array(val_labels)) if val_labels else float('nan')

        print(f"[{organ}] Epoch {epoch+1}: train_loss={train_loss:.4f}  val_loss={val_loss:.4f}  val_acc={val_acc:.3f}")
        history.append({
            "organ": organ, "epoch": epoch+1,
            "train_loss": train_loss, "val_loss": val_loss, "val_acc": val_acc
        })

        if val_loss < best_val_loss:
            best_val_loss = val_loss
            torch.save(model.state_dict(), OUTPUT_DIR / f"best_{organ}_model.pt")
            best_val_predictions = list(zip(val_labels, val_preds))

    # Save per-sample val predictions from the best epoch, for manual inspection
    pred_df = pd.DataFrame(best_val_predictions, columns=["true_label", "pred_label"])
    pred_df.to_csv(OUTPUT_DIR / f"{organ}_val_predictions.csv", index=False)

    return model, history

all_history = []
for organ in ["liver", "spleen", "kidney", "bowel"]:
    print(f"\n=== Training baseline: {organ} ===")
    _, hist = train_organ_model(organ, epochs=5)
    all_history.extend(hist)

history_df = pd.DataFrame(all_history)
history_df.to_csv(OUTPUT_DIR / "baseline_training_history.csv", index=False)

print("\nFinal baseline val_loss per organ (best epoch):")
best_rows = history_df.loc[history_df.groupby('organ')['val_loss'].idxmin()]
print(best_rows[['organ', 'epoch', 'val_loss', 'val_acc']])

Using device: cuda

=== Training baseline: liver ===
[liver] train samples: 173, val samples: 33


model.safetensors:   0%|          | 0.00/21.4M [00:00<?, ?B/s]

liver epoch 1/5: 100%|██████████| 44/44 [00:31<00:00,  1.38it/s]


[liver] Epoch 1: train_loss=2.0808  val_loss=1.8341  val_acc=0.455


liver epoch 2/5: 100%|██████████| 44/44 [00:31<00:00,  1.40it/s]


...

[bowel] Epoch 3: train_loss=0.2968  val_loss=0.3881  val_acc=0.939


bowel epoch 4/5: 100%|██████████| 44/44 [00:38<00:00,  1.14it/s]


[bowel] Epoch 4: train_loss=0.2036  val_loss=0.3010  val_acc=0.939


bowel epoch 5/5: 100%|██████████| 44/44 [00:38<00:00,  1.15it/s]


[bowel] Epoch 5: train_loss=0.1877  val_loss=0.2898  val_acc=0.848

Final baseline val_loss per organ (best epoch):
     organ  epoch  val_loss   val_acc
16   bowel      2  0.214619  0.969697
13  kidney      4  1.174467  0.696970
1    liver      2  1.716891  0.515152
6   spleen      2  1.672094  0.606061


In [10]:
bowel_preds = pd.read_csv(OUTPUT_DIR / "bowel_val_predictions.csv")
print(bowel_preds)
print("\nPrediction distribution:")
print(bowel_preds['pred_label'].value_counts())
print("\nDid it catch the one positive case?")
print(bowel_preds[bowel_preds['true_label'] == 1])

    true_label  pred_label
0            0           0
1            0           0
2            0           0
3            0           0
4            0           0
5            0           0
6            0           0
7            0           0
8            0           0
9            0           0
10           0           0
11           0           0
12           0           0
13           0           0
14           0           0
15           0           0
16           0           0
17           0           0
18           1           0
19           0           0
20           0           0
21           0           0
22           0           0
23           0           0
24           0           0
25           0           0
26           0           0
27           0           0
28           0           0
29           0           0
30           0           0
31           0           0
32           0           0

Prediction distribution:
pred_label
0    33
Name: count, dtype: int64

Did it cat

In [11]:
from sklearn.metrics import classification_report

for organ in ["liver", "spleen", "kidney", "bowel"]:
    preds = pd.read_csv(OUTPUT_DIR / f"{organ}_val_predictions.csv")
    print(f"\n=== {organ} ===")
    print(classification_report(preds['true_label'], preds['pred_label'], zero_division=0))


=== liver ===
              precision    recall  f1-score   support

           0       0.50      0.14      0.22        14
           1       0.48      0.87      0.62        15
           2       1.00      0.50      0.67         4

    accuracy                           0.52        33
   macro avg       0.66      0.50      0.50        33
weighted avg       0.55      0.52      0.46        33


=== spleen ===
              precision    recall  f1-score   support

           0       1.00      0.31      0.47        13
           1       0.36      0.57      0.44         7
           2       0.67      0.92      0.77        13

    accuracy                           0.61        33
   macro avg       0.68      0.60      0.56        33
weighted avg       0.73      0.61      0.58        33


=== kidney ===
              precision    recall  f1-score   support

           0       1.00      0.53      0.70        15
           1       0.62      0.71      0.67         7
           2       0.59     

In [12]:
from sklearn.metrics import classification_report
import json

SEEDS = [42, 123, 2024, 7, 555]
ORGANS_FOR_MULTISEED = ["liver", "spleen", "kidney"]  # bowel excluded — degenerate regardless of seed

multiseed_results = []

for organ in ORGANS_FOR_MULTISEED:
    print(f"\n{'='*50}")
    print(f"Multi-seed evaluation: {organ}")
    print(f"{'='*50}")

    for seed in SEEDS:
        print(f"\n--- {organ}, seed={seed} ---")
        _, hist = train_organ_model(organ, epochs=5, seed=seed)

        # Find best epoch for this run
        hist_df = pd.DataFrame(hist)
        best_row = hist_df.loc[hist_df['val_loss'].idxmin()]

        # Load the saved best-epoch predictions for per-class metrics
        pred_df = pd.read_csv(OUTPUT_DIR / f"{organ}_val_predictions.csv")
        report = classification_report(
            pred_df['true_label'], pred_df['pred_label'],
            output_dict=True, zero_division=0
        )

        multiseed_results.append({
            "organ": organ,
            "seed": seed,
            "best_epoch": int(best_row['epoch']),
            "val_loss": best_row['val_loss'],
            "val_acc": best_row['val_acc'],
            "per_class_recall": json.dumps({k: v['recall'] for k, v in report.items() if k.isdigit()}),
        })

multiseed_df = pd.DataFrame(multiseed_results)
multiseed_df.to_csv(OUTPUT_DIR / "multiseed_baseline_results.csv", index=False)

print("\n\n" + "="*50)
print("SUMMARY: Mean ± Std across seeds")
print("="*50)
for organ in ORGANS_FOR_MULTISEED:
    sub = multiseed_df[multiseed_df['organ'] == organ]
    print(f"\n{organ}:")
    print(f"  val_loss: {sub['val_loss'].mean():.4f} ± {sub['val_loss'].std():.4f}")
    print(f"  val_acc:  {sub['val_acc'].mean():.3f} ± {sub['val_acc'].std():.3f}")
    print(f"  individual runs: {sub['val_loss'].round(4).tolist()}")


Multi-seed evaluation: liver

--- liver, seed=42 ---
[liver] train samples: 173, val samples: 33


liver epoch 1/5: 100%|██████████| 44/44 [00:38<00:00,  1.16it/s]


[liver] Epoch 1: train_loss=2.0808  val_loss=1.8341  val_acc=0.455


liver epoch 2/5: 100%|██████████| 44/44 [00:38<00:00,  1.15it/s]


[liver] Epoch 2: train_loss=1.7253  val_loss=1.7169  val_acc=0.515


...

[kidney] Epoch 3: train_loss=1.3415  val_loss=1.7180  val_acc=0.697


kidney epoch 4/5: 100%|██████████| 43/43 [00:37<00:00,  1.14it/s]


[kidney] Epoch 4: train_loss=0.5801  val_loss=1.3085  val_acc=0.758


kidney epoch 5/5: 100%|██████████| 43/43 [00:37<00:00,  1.15it/s]


[kidney] Epoch 5: train_loss=0.2408  val_loss=1.6613  val_acc=0.606


SUMMARY: Mean ± Std across seeds

liver:
  val_loss: 1.8429 ± 0.0817
  val_acc:  0.406 ± 0.119
  individual runs: [1.7169, 1.805, 1.9147, 1.8885, 1.8896]

spleen:
  val_loss: 1.5318 ± 0.1330
  val_acc:  0.527 ± 0.051
  individual runs: [1.6721, 1.4577, 1.5256, 1.6498, 1.3538]

kidney:
  val_loss: 1.3180 ± 0.1194
  val_acc:  0.703 ± 0.062
  individual runs: [1.1745, 1.5067, 1.2945, 1.306, 1.3085]


In [13]:
print("Per-class recall, mean ± std across seeds:\n")

for organ in ORGANS_FOR_MULTISEED:
    sub = multiseed_df[multiseed_df['organ'] == organ]
    recalls_per_seed = sub['per_class_recall'].apply(json.loads)

    # Collect recall for each class across all seeds
    all_classes = set()
    for r in recalls_per_seed:
        all_classes.update(r.keys())

    print(f"{organ}:")
    for cls in sorted(all_classes):
        vals = [r.get(cls, np.nan) for r in recalls_per_seed]
        vals = [v for v in vals if not np.isnan(v)]
        if vals:
            print(f"  class {cls} recall: {np.mean(vals):.3f} ± {np.std(vals):.3f}  (n_seeds_with_class={len(vals)})")
    print()

Per-class recall, mean ± std across seeds:

liver:
  class 0 recall: 0.143 ± 0.045  (n_seeds_with_class=5)
  class 1 recall: 0.653 ± 0.212  (n_seeds_with_class=5)
  class 2 recall: 0.400 ± 0.255  (n_seeds_with_class=5)

spleen:
  class 0 recall: 0.092 ± 0.123  (n_seeds_with_class=5)
  class 1 recall: 0.514 ± 0.070  (n_seeds_with_class=5)
  class 2 recall: 0.969 ± 0.038  (n_seeds_with_class=5)

kidney:
  class 0 recall: 0.600 ± 0.094  (n_seeds_with_class=5)
  class 1 recall: 0.686 ± 0.107  (n_seeds_with_class=5)
  class 2 recall: 0.855 ± 0.045  (n_seeds_with_class=5)



In [14]:
class OrdinalPresentLabelUpweightingLoss(nn.Module):
    """
    Generalizes present-label loss upweighting (binary presence/absence in
    prior aneurysm detection work) to ordinal severity grading.

    - Effective-number class balancing counters the healthy-recall collapse
      seen in the weighted-CE baseline.
    - Ordinal distance penalty respects healthy < low < high structure.
    - Present-label boost preserves injury sensitivity independent of the
      class-balancing term.
    """
    def __init__(self, class_counts, beta=0.999, ordinal_alpha=0.5,
                 present_boost=1.5, focal_gamma=1.0):
        super().__init__()
        class_counts = np.array(class_counts, dtype=np.float32)
        effective_num = 1.0 - np.power(beta, class_counts)
        weights = (1.0 - beta) / np.clip(effective_num, 1e-8, None)
        weights = weights / weights.sum() * len(class_counts)
        self.class_weights = torch.tensor(weights, dtype=torch.float32)

        self.ordinal_alpha = ordinal_alpha
        self.present_boost = present_boost
        self.focal_gamma = focal_gamma
        self.num_classes = len(class_counts)

    def forward(self, logits, targets, sample_weight=None):
        device = logits.device
        class_weights = self.class_weights.to(device)

        probs = torch.softmax(logits, dim=1)
        log_probs = torch.log_softmax(logits, dim=1)

        target_probs = probs.gather(1, targets.unsqueeze(1)).squeeze(1)
        focal_factor = (1 - target_probs) ** self.focal_gamma
        ce = -log_probs.gather(1, targets.unsqueeze(1)).squeeze(1)
        cls_w = class_weights[targets]
        base_loss = focal_factor * cls_w * ce

        class_ranks = torch.arange(self.num_classes, device=device).float()
        expected_rank = (probs * class_ranks.unsqueeze(0)).sum(dim=1)
        ordinal_penalty = (expected_rank - targets.float()) ** 2

        present_mask = (targets > 0).float()
        present_weight = 1.0 + (self.present_boost - 1.0) * present_mask

        loss = present_weight * (base_loss + self.ordinal_alpha * ordinal_penalty)

        if sample_weight is not None:
            loss = loss * sample_weight

        return loss.mean()

In [15]:
def get_class_counts(manifest_df, organ, split='train'):
    sub = manifest_df[(manifest_df['organs_cropped'].apply(lambda x: organ in x)) & (manifest_df['split'] == split)]
    labels = sub.apply(lambda row: get_organ_label(row, organ), axis=1)
    counts = [np.sum(labels == c) for c in range(ORGAN_NUM_CLASSES[organ])]
    return counts

def train_organ_model_ordinal(organ, epochs=5, lr=1e-4, seed=42,
                                ordinal_alpha=0.5, present_boost=1.5, focal_gamma=1.0):
    set_seed(seed)

    train_loader = make_loader(organ_manifest, organ, 'train')
    val_loader = make_loader(organ_manifest, organ, 'val')

    class_counts = get_class_counts(organ_manifest, organ, 'train')
    print(f"[{organ}] train samples: {len(train_loader.dataset)}, val samples: {len(val_loader.dataset)}, class_counts: {class_counts}")

    model = OrganClassifier(num_classes=ORGAN_NUM_CLASSES[organ]).to(device)
    optimizer = optim.AdamW(model.parameters(), lr=lr)
    criterion = OrdinalPresentLabelUpweightingLoss(
        class_counts, ordinal_alpha=ordinal_alpha,
        present_boost=present_boost, focal_gamma=focal_gamma
    )

    best_val_loss = float('inf')
    history = []
    best_val_predictions = None

    for epoch in range(epochs):
        model.train()
        train_losses = []
        for vol, label, weight in tqdm(train_loader, desc=f"[ordinal] {organ} epoch {epoch+1}/{epochs}"):
            vol, label = vol.to(device), label.to(device)
            optimizer.zero_grad()
            logits = model(vol)
            loss = criterion(logits, label)
            loss.backward()
            optimizer.step()
            train_losses.append(loss.item())

        model.eval()
        val_losses = []
        val_preds, val_labels = [], []
        with torch.no_grad():
            for vol, label, weight in val_loader:
                vol, label = vol.to(device), label.to(device)
                logits = model(vol)
                loss = criterion(logits, label)
                val_losses.append(loss.item())
                val_preds.extend(torch.argmax(logits, dim=1).cpu().numpy().tolist())
                val_labels.extend(label.cpu().numpy().tolist())

        train_loss = np.mean(train_losses) if train_losses else float('nan')
        val_loss = np.mean(val_losses) if val_losses else float('nan')
        val_acc = np.mean(np.array(val_preds) == np.array(val_labels)) if val_labels else float('nan')

        print(f"[ordinal][{organ}] Epoch {epoch+1}: train_loss={train_loss:.4f}  val_loss={val_loss:.4f}  val_acc={val_acc:.3f}")
        history.append({
            "organ": organ, "epoch": epoch+1,
            "train_loss": train_loss, "val_loss": val_loss, "val_acc": val_acc
        })

        if val_loss < best_val_loss:
            best_val_loss = val_loss
            torch.save(model.state_dict(), OUTPUT_DIR / f"best_{organ}_ordinal_model.pt")
            best_val_predictions = list(zip(val_labels, val_preds))

    pred_df = pd.DataFrame(best_val_predictions, columns=["true_label", "pred_label"])
    pred_df.to_csv(OUTPUT_DIR / f"{organ}_ordinal_val_predictions.csv", index=False)

    return model, history

In [16]:
ordinal_multiseed_results = []

for organ in ORGANS_FOR_MULTISEED:
    print(f"\n{'='*50}")
    print(f"Multi-seed ORDINAL evaluation: {organ}")
    print(f"{'='*50}")

    for seed in SEEDS:
        print(f"\n--- [ordinal] {organ}, seed={seed} ---")
        _, hist = train_organ_model_ordinal(organ, epochs=5, seed=seed)

        hist_df = pd.DataFrame(hist)
        best_row = hist_df.loc[hist_df['val_loss'].idxmin()]

        pred_df = pd.read_csv(OUTPUT_DIR / f"{organ}_ordinal_val_predictions.csv")
        report = classification_report(
            pred_df['true_label'], pred_df['pred_label'],
            output_dict=True, zero_division=0
        )

        ordinal_multiseed_results.append({
            "organ": organ, "seed": seed,
            "best_epoch": int(best_row['epoch']),
            "val_loss": best_row['val_loss'],
            "val_acc": best_row['val_acc'],
            "per_class_recall": json.dumps({k: v['recall'] for k, v in report.items() if k.isdigit()}),
        })

ordinal_multiseed_df = pd.DataFrame(ordinal_multiseed_results)
ordinal_multiseed_df.to_csv(OUTPUT_DIR / "multiseed_ordinal_results.csv", index=False)

print("\n\n" + "="*60)
print("BASELINE vs ORDINAL — Mean ± Std across seeds")
print("="*60)
for organ in ORGANS_FOR_MULTISEED:
    base = multiseed_df[multiseed_df['organ'] == organ]
    ordn = ordinal_multiseed_df[ordinal_multiseed_df['organ'] == organ]
    print(f"\n{organ}:")
    print(f"  BASELINE  val_loss={base['val_loss'].mean():.4f}±{base['val_loss'].std():.4f}  val_acc={base['val_acc'].mean():.3f}±{base['val_acc'].std():.3f}")
    print(f"  ORDINAL   val_loss={ordn['val_loss'].mean():.4f}±{ordn['val_loss'].std():.4f}  val_acc={ordn['val_acc'].mean():.3f}±{ordn['val_acc'].std():.3f}")

print("\n\nPer-class recall comparison:")
for organ in ORGANS_FOR_MULTISEED:
    print(f"\n{organ}:")
    base_recalls = multiseed_df[multiseed_df['organ']==organ]['per_class_recall'].apply(json.loads)
    ordn_recalls = ordinal_multiseed_df[ordinal_multiseed_df['organ']==organ]['per_class_recall'].apply(json.loads)
    all_classes = sorted(set().union(*[set(r.keys()) for r in base_recalls]))
    for cls in all_classes:
        b_vals = [r.get(cls, np.nan) for r in base_recalls]
        o_vals = [r.get(cls, np.nan) for r in ordn_recalls]
        b_vals = [v for v in b_vals if not np.isnan(v)]
        o_vals = [v for v in o_vals if not np.isnan(v)]
        print(f"  class {cls}:  baseline={np.mean(b_vals):.3f}±{np.std(b_vals):.3f}   ordinal={np.mean(o_vals):.3f}±{np.std(o_vals):.3f}")


Multi-seed ORDINAL evaluation: liver

--- [ordinal] liver, seed=42 ---
[liver] train samples: 173, val samples: 33, class_counts: [np.int64(69), np.int64(77), np.int64(27)]


[ordinal] liver epoch 1/5: 100%|██████████| 44/44 [00:38<00:00,  1.14it/s]


[ordinal][liver] Epoch 1: train_loss=1.1246  val_loss=0.9871  val_acc=0.455


[ordinal] liver epoch 2/5: 100%|██████████| 44/44 [00:38<00:00,  1.16it/s]


[ordinal][liver] Epoch 2: train_loss=0.9013  val_loss=0.9608  val_acc=0.485


...

[ordinal][kidney] Epoch 3: train_loss=0.7287  val_loss=0.7117  val_acc=0.636


[ordinal] kidney epoch 4/5: 100%|██████████| 43/43 [00:38<00:00,  1.13it/s]


[ordinal][kidney] Epoch 4: train_loss=0.2725  val_loss=0.4944  val_acc=0.818


[ordinal] kidney epoch 5/5: 100%|██████████| 43/43 [00:38<00:00,  1.13it/s]


[ordinal][kidney] Epoch 5: train_loss=0.2016  val_loss=0.7398  val_acc=0.758


BASELINE vs ORDINAL — Mean ± Std across seeds

liver:
  BASELINE  val_loss=1.8429±0.0817  val_acc=0.406±0.119
  ORDINAL   val_loss=0.9818±0.0406  val_acc=0.467±0.059

spleen:
  BASELINE  val_loss=1.5318±0.1330  val_acc=0.527±0.051
  ORDINAL   val_loss=0.9391±0.0703  val_acc=0.582±0.058

kidney:
  BASELINE  val_loss=1.3180±0.1194  val_acc=0.703±0.062
  ORDINAL   val_loss=0.6546±0.1168  val_acc=0.709±0.070


Per-class recall comparison:

liver:
  class 0:  baseline=0.143±0.045   ordinal=0.300±0.083
  class 1:  baseline=0.653±0.212   ordinal=0.600±0.174
  class 2:  baseline=0.400±0.255   ordinal=0.550±0.100

spleen:
  class 0:  baseline=0.092±0.123   ordinal=0.338±0.078
  class 1:  baseline=0.514±0.070   ordinal=0.514±0.146
  class 2:  baseline=0.969±0.038   ordinal=0.862±0.075

kidney:
  class 0:  baseline=0.600±0.094   ordinal=0.680±0.050
  class 1:  baseline=0.686±0.107   ordinal=0.514±0.146
  class 2:  base

In [17]:
import itertools

ALPHA_GRID = [0.1, 0.5, 1.0]
BOOST_GRID = [1.0, 1.5, 2.0]  # 1.0 = no present-label boost, isolates ordinal_alpha's effect
SWEEP_ORGANS = ["kidney", "spleen"]
SWEEP_SEED = 42  # single seed for the grid; winner gets multi-seed confirmation after

sweep_results = []

for organ in SWEEP_ORGANS:
    print(f"\n{'='*60}")
    print(f"Hyperparameter sweep: {organ}")
    print(f"{'='*60}")

    for alpha, boost in itertools.product(ALPHA_GRID, BOOST_GRID):
        print(f"\n--- {organ}: alpha={alpha}, boost={boost} ---")
        _, hist = train_organ_model_ordinal(
            organ, epochs=5, seed=SWEEP_SEED,
            ordinal_alpha=alpha, present_boost=boost
        )

        hist_df = pd.DataFrame(hist)
        best_row = hist_df.loc[hist_df['val_loss'].idxmin()]

        pred_df = pd.read_csv(OUTPUT_DIR / f"{organ}_ordinal_val_predictions.csv")
        report = classification_report(
            pred_df['true_label'], pred_df['pred_label'],
            output_dict=True, zero_division=0
        )
        per_class_recall = {k: v['recall'] for k, v in report.items() if k.isdigit()}

        sweep_results.append({
            "organ": organ,
            "ordinal_alpha": alpha,
            "present_boost": boost,
            "val_loss": best_row['val_loss'],
            "val_acc": best_row['val_acc'],
            "recall_class0": per_class_recall.get('0', np.nan),
            "recall_class1": per_class_recall.get('1', np.nan),
            "recall_class2": per_class_recall.get('2', np.nan),
        })

sweep_df = pd.DataFrame(sweep_results)
sweep_df.to_csv(OUTPUT_DIR / "hyperparameter_sweep_results.csv", index=False)

print("\n\n" + "="*70)
print("SWEEP RESULTS")
print("="*70)
for organ in SWEEP_ORGANS:
    print(f"\n{organ}:")
    sub = sweep_df[sweep_df['organ'] == organ].sort_values(['ordinal_alpha', 'present_boost'])
    print(sub[['ordinal_alpha', 'present_boost', 'val_acc', 'recall_class0', 'recall_class1', 'recall_class2']].to_string(index=False))


Hyperparameter sweep: kidney

--- kidney: alpha=0.1, boost=1.0 ---
[kidney] train samples: 171, val samples: 33, class_counts: [np.int64(86), np.int64(49), np.int64(36)]


[ordinal] kidney epoch 1/5: 100%|██████████| 43/43 [00:38<00:00,  1.13it/s]


[ordinal][kidney] Epoch 1: train_loss=0.6888  val_loss=0.8112  val_acc=0.212


[ordinal] kidney epoch 2/5: 100%|██████████| 43/43 [00:37<00:00,  1.14it/s]


[ordinal][kidney] Epoch 2: train_loss=0.5509  val_loss=0.5827  val_acc=0.606


...

[ordinal][spleen] Epoch 3: train_loss=1.4952  val_loss=1.7382  val_acc=0.545


[ordinal] spleen epoch 4/5: 100%|██████████| 44/44 [00:38<00:00,  1.16it/s]


[ordinal][spleen] Epoch 4: train_loss=1.0606  val_loss=1.6932  val_acc=0.545


[ordinal] spleen epoch 5/5: 100%|██████████| 44/44 [00:37<00:00,  1.16it/s]


[ordinal][spleen] Epoch 5: train_loss=0.6712  val_loss=1.9456  val_acc=0.455


SWEEP RESULTS

kidney:
 ordinal_alpha  present_boost  val_acc  recall_class0  recall_class1  recall_class2
           0.1            1.0 0.666667       0.800000       0.285714       0.727273
           0.1            1.5 0.696970       0.733333       0.285714       0.909091
           0.1            2.0 0.696970       0.666667       0.428571       0.909091
           0.5            1.0 0.696970       0.800000       0.428571       0.727273
           0.5            1.5 0.696970       0.666667       0.428571       0.909091
           0.5            2.0 0.727273       0.666667       0.571429       0.909091
           1.0            1.0 0.727273       0.600000       0.714286       0.909091
           1.0            1.5 0.636364       0.666667       0.571429       0.636364
           1.0            2.0 0.666667       0.666667       0.428571       0.818182

spleen:
 ordinal_alpha  present_boost  val_acc  recall_cl

In [18]:
def score_config(row, organ):
    # Weighted score: overall accuracy, but penalize the specific regression flagged earlier.
    # Kidney: penalize low class-1 recall (the observed regression).
    # Spleen: penalize low class-2 recall (the observed regression).
    if organ == "kidney":
        return 0.5 * row['val_acc'] + 0.3 * row['recall_class1'] + 0.2 * row['recall_class0']
    elif organ == "spleen":
        return 0.5 * row['val_acc'] + 0.3 * row['recall_class2'] + 0.2 * row['recall_class0']
    return row['val_acc']

for organ in SWEEP_ORGANS:
    sub = sweep_df[sweep_df['organ'] == organ].copy()
    sub['composite_score'] = sub.apply(lambda r: score_config(r, organ), axis=1)
    best = sub.loc[sub['composite_score'].idxmax()]
    print(f"\n{organ} — best config by composite score:")
    print(f"  ordinal_alpha={best['ordinal_alpha']}, present_boost={best['present_boost']}")
    print(f"  val_acc={best['val_acc']:.3f}, recall_c0={best['recall_class0']:.3f}, "
          f"recall_c1={best['recall_class1']:.3f}, recall_c2={best['recall_class2']:.3f}")


kidney — best config by composite score:
  ordinal_alpha=1.0, present_boost=1.0
  val_acc=0.727, recall_c0=0.600, recall_c1=0.714, recall_c2=0.909

spleen — best config by composite score:
  ordinal_alpha=1.0, present_boost=1.0
  val_acc=0.667, recall_c0=0.615, recall_c1=0.571, recall_c2=0.769


In [19]:
# UPDATE these based on Cell 15's output before running
BEST_CONFIGS = {
    "kidney": {"ordinal_alpha": None, "present_boost": None},  # fill in from Cell 15
    "spleen": {"ordinal_alpha": None, "present_boost": None},  # fill in from Cell 15
}

confirmed_results = []

for organ, cfg in BEST_CONFIGS.items():
    if cfg["ordinal_alpha"] is None:
        print(f"Skipping {organ} — fill in BEST_CONFIGS first")
        continue

    print(f"\n{'='*50}")
    print(f"Confirming best config for {organ}: {cfg}")
    print(f"{'='*50}")

    for seed in SEEDS:
        _, hist = train_organ_model_ordinal(organ, epochs=5, seed=seed, **cfg)
        hist_df = pd.DataFrame(hist)
        best_row = hist_df.loc[hist_df['val_loss'].idxmin()]

        pred_df = pd.read_csv(OUTPUT_DIR / f"{organ}_ordinal_val_predictions.csv")
        report = classification_report(pred_df['true_label'], pred_df['pred_label'], output_dict=True, zero_division=0)

        confirmed_results.append({
            "organ": organ, "seed": seed,
            "val_loss": best_row['val_loss'], "val_acc": best_row['val_acc'],
            "per_class_recall": json.dumps({k: v['recall'] for k, v in report.items() if k.isdigit()}),
        })

confirmed_df = pd.DataFrame(confirmed_results)
confirmed_df.to_csv(OUTPUT_DIR / "confirmed_tuned_results.csv", index=False)

print("\nTuned config, mean ± std across seeds:")
for organ in BEST_CONFIGS:
    if BEST_CONFIGS[organ]["ordinal_alpha"] is None:
        continue
    sub = confirmed_df[confirmed_df['organ'] == organ]
    print(f"\n{organ} (alpha={BEST_CONFIGS[organ]['ordinal_alpha']}, boost={BEST_CONFIGS[organ]['present_boost']}):")
    print(f"  val_acc: {sub['val_acc'].mean():.3f} ± {sub['val_acc'].std():.3f}")

Skipping kidney — fill in BEST_CONFIGS first
Skipping spleen — fill in BEST_CONFIGS first

Tuned config, mean ± std across seeds:


In [20]:
ALPHA_GRID_EXTENDED = [1.0, 1.5, 2.0, 3.0]
FIXED_BOOST = 1.0

extended_results = []

for organ in SWEEP_ORGANS:
    print(f"\n=== Extended alpha sweep: {organ} (boost fixed at {FIXED_BOOST}) ===")
    for alpha in ALPHA_GRID_EXTENDED:
        print(f"\n--- {organ}: alpha={alpha} ---")
        _, hist = train_organ_model_ordinal(organ, epochs=5, seed=SWEEP_SEED,
                                              ordinal_alpha=alpha, present_boost=FIXED_BOOST)
        hist_df = pd.DataFrame(hist)
        best_row = hist_df.loc[hist_df['val_loss'].idxmin()]
        pred_df = pd.read_csv(OUTPUT_DIR / f"{organ}_ordinal_val_predictions.csv")
        report = classification_report(pred_df['true_label'], pred_df['pred_label'], output_dict=True, zero_division=0)
        per_class_recall = {k: v['recall'] for k, v in report.items() if k.isdigit()}

        extended_results.append({
            "organ": organ, "ordinal_alpha": alpha, "present_boost": FIXED_BOOST,
            "val_acc": best_row['val_acc'],
            "recall_class0": per_class_recall.get('0', np.nan),
            "recall_class1": per_class_recall.get('1', np.nan),
            "recall_class2": per_class_recall.get('2', np.nan),
        })

extended_df = pd.DataFrame(extended_results)
print("\n\nExtended sweep results:")
for organ in SWEEP_ORGANS:
    print(f"\n{organ}:")
    print(extended_df[extended_df['organ']==organ].to_string(index=False))


=== Extended alpha sweep: kidney (boost fixed at 1.0) ===

--- kidney: alpha=1.0 ---
[kidney] train samples: 171, val samples: 33, class_counts: [np.int64(86), np.int64(49), np.int64(36)]


[ordinal] kidney epoch 1/5: 100%|██████████| 43/43 [00:38<00:00,  1.13it/s]


[ordinal][kidney] Epoch 1: train_loss=1.2452  val_loss=1.4272  val_acc=0.212


[ordinal] kidney epoch 2/5: 100%|██████████| 43/43 [00:37<00:00,  1.15it/s]


[ordinal][kidney] Epoch 2: train_loss=0.9557  val_loss=0.9674  val_acc=0.727


...

[ordinal][spleen] Epoch 3: train_loss=1.8734  val_loss=2.1641  val_acc=0.727


[ordinal] spleen epoch 4/5: 100%|██████████| 44/44 [00:38<00:00,  1.15it/s]


[ordinal][spleen] Epoch 4: train_loss=1.0296  val_loss=2.5842  val_acc=0.545


[ordinal] spleen epoch 5/5: 100%|██████████| 44/44 [00:37<00:00,  1.16it/s]


[ordinal][spleen] Epoch 5: train_loss=0.6302  val_loss=2.5721  val_acc=0.545


Extended sweep results:

kidney:
 organ  ordinal_alpha  present_boost  val_acc  recall_class0  recall_class1  recall_class2
kidney            1.0            1.0 0.727273       0.600000       0.714286       0.909091
kidney            1.5            1.0 0.727273       0.600000       0.714286       0.909091
kidney            2.0            1.0 0.696970       0.533333       0.714286       0.909091
kidney            3.0            1.0 0.696970       0.533333       0.714286       0.909091

spleen:
 organ  ordinal_alpha  present_boost  val_acc  recall_class0  recall_class1  recall_class2
spleen            1.0            1.0 0.666667       0.615385       0.571429       0.769231
spleen            1.5            1.0 0.696970       0.692308       0.571429       0.769231
spleen            2.0            1.0 0.727273       0.769231       0.571429       0.769231
spleen            3.0            1.0 0.757576       0.769231